# Yahoo Finance ('yfinance') Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [15]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [16]:
# Let's create a list of sectors
sector_etfs = [
    # Broad equity
    "SPY", "QQQ", "IWM",
    # GICS sectors
    "XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY",
    # Fixed income — short / intermediate / long / credit / inflation
    "BIL", "BND", "IEF", "TLT", "LQD", "HYG", "TIP",
    # Real assets
    "XLRE", "GLD"
]
# price data
prices = yf.download(sector_etfs, period='max', auto_adjust=True)['Close']

[*********************100%***********************]  21 of 21 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [17]:

prices.head()



Ticker,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.241407,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.413809,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.465544,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.724165,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.827606,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
prices.tail()

Ticker,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-05,91.419998,74.339996,466.130005,80.080002,96.510002,256.760010,110.529999,608.909973,681.309998,111.220001,...,50.830002,56.480000,51.230000,172.059998,140.179993,85.410004,43.340000,46.900002,153.910004,116.550003
2026-03-06,91.440002,74.239998,473.510010,79.690002,96.449997,250.889999,110.160004,599.750000,672.380005,111.430000,...,49.860001,56.570000,50.570000,169.940002,137.289993,85.779999,42.889999,46.740002,152.699997,114.440002
2026-03-09,91.449997,74.449997,472.529999,80.169998,96.750000,253.619995,110.820000,607.760010,678.270020,111.610001,...,49.990002,56.320000,50.330002,170.940002,139.759995,85.970001,42.980000,46.849998,154.259995,114.589996
2026-03-10,91.459999,74.220001,477.859985,80.040001,96.440002,253.360001,110.059998,607.770020,677.179993,111.239998,...,49.880001,55.599998,50.060001,170.020004,139.759995,85.720001,42.919998,46.560001,153.149994,114.440002
2026-03-11,91.459999,73.934998,474.769989,79.885002,96.074997,251.535004,109.285004,606.789978,674.789978,111.125000,...,49.439999,56.485001,49.375000,169.240005,140.089996,84.529999,42.509998,46.329899,152.600006,113.940002


Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [19]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 1993-01-29 00:00:00.


Ryan Peet brought up a really good idea of using mutual funds as proxies as they have been investment vehicles for a much longer period of time. So let's see if we can build the same datapull for mutual funds, and see how far back that data goes.  
Broad Market (SPY proxy): Vanguard 500 Index (VFINX) — data back to 1976, the gold standard  
Tech (XLK): Fidelity Select Technology (FSPTX) — inception 1981  
Healthcare (XLV): Fidelity Select Health Care (FSPHX) — inception 1981  
Energy (XLE): Fidelity Select Energy (FSENX) — inception 1981  
Financials (XLF): Fidelity Select Financial Services (FIDSX) — inception 1981  
Utilities (XLU): Fidelity Select Utilities (FSUTX) — inception 1981  
*Note* Industrials is really hard because it wasn't really a sector until the late 90's early '00s. It may be best to just drop it as the only one that works is a very heavily weighted subsection of industrials  
Industrials (XLI): Fidelity Select Industrials (FCYIX) — inception 1997 (this one is shorter I actually could only get data to 2019)
Industrials2 (XLI): Fidelity Select Defense & Aerospace (FSDAX)  
Consumer Staples (XLP): Fidelity Select Consumer Staples (FDFAX) — inception 1985  
Consumer Discretionary (XLY): Fidelity Select Retailing (FSRPX) as an imperfect proxy  
Materials (XLB): Fidelity Select Materials (FSDPX) — inception 1986  
Bonds (short-term): Vanguard Short-Term Bond Index (VBISX) or use direct Treasury yields from FRED  
Bonds (long-term): Vanguard Long-Term Bond Index (VBLTX) or TLT equivalent via Barclays index data from FRED  
Money market: 3-month T-bill rate from FRED is cleaner than any fund proxy  


In [20]:
# Let's create a list of sectors
sector_mfs = ["VFINX","FSPTX","FSPHX","FSENX","FIDSX","FSUTX","FSDAX","FDFAX","FSRPX","FSDPX","VBISX","VBLTX"]
# I am going to comment out the mutual fund pull because we decided against them
#prices_mfs = yf.download(sector_mfs, period='max', auto_adjust=True)[['Close','Volume']]

In [21]:
#prices_mfs.dropna(inplace=True)
#prices_mfs.head()



In [22]:
#prices_mfs.tail()

Alright, based on this analysis, I still think that the ETF's are the strongest approach. Let's also look at some different indices that may be beneficial to our understanding of the current environment (independent variables).  

Volatility & Fear  
^VIX — CBOE Volatility Index (equity fear gauge)  
^VXN — Nasdaq volatility equivalent  
^MOVE — Bond market volatility (the "VIX for Treasuries")  

Currency  

DX-Y.NYB — DXY Dollar Index  
EURUSD=X, JPYUSD=X, CNYUSD=X — Major pairs (EUR, Yen, Yuan signal global risk appetite and trade conditions)  

Rates & Credit  

^TNX — 10-Year Treasury yield  
^TYX — 30-Year Treasury yield  
^IRX — 13-week T-Bill (short end)  
^FVX — 5-Year Treasury yield  

Commodities (macro signals)

GC=F — Gold (inflation hedge / flight to safety)  
CL=F — Crude Oil WTI (growth proxy, geopolitical risk)  
NG=F — Natural Gas  
HG=F — Copper ("Dr. Copper" — leading economic indicator)  

Credit Spreads (via ETFs since yfinance doesn't have spread data directly)  

HYG — High Yield Corporate Bonds (risk appetite)  
LQD — Investment Grade Corporate Bonds  
TLT — Long Duration Treasuries (rate sensitivity)  
SHY — Short Duration Treasuries  

Global / Geopolitical

^FTSE — UK (Brexit/European stability)  
^N225 — Nikkei (Japan / Asia Pacific) #Not working so lets try futures
NKY=F - Nikkei Futures (not spot price) Also not working so lets remove   
^HSI — Hang Seng (China exposure)  
^GSPC — S&P 500 broad market  

In [23]:
indicies = ["^VIX","^VXN","^MOVE","DX-Y.NYB","^TNX","GC=F","CL=F","NG=F","HG=F","HYG","LQD","TLT","SHY","^FTSE","^HSI","^GSPC"] # When we removed the Credit spreads we went back to 2002, If we remove the VIX and VXN and MOVE then we can go back to late 2000.

prices_indicies = yf.download(indicies, period='max', auto_adjust=True)['Close']
# We may need to run the data with more historical data but less features and more features but less historical data to see what works best for our model but keep in mind the impact that it will have on out of sample data.

[*********************100%***********************]  16 of 16 completed


In [24]:
#prices_indicies.dropna(inplace=True)
prices_indicies.head()

Ticker,CL=F,DX-Y.NYB,GC=F,HG=F,HYG,LQD,NG=F,SHY,TLT,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
Date,,,,,,,,,,,,,,,,
1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


This amount of data only goes back to 2007, so we may want to consider dropping some of them to see if we can get data back to our origination of the macro and ETF data (around 1994)

Let's merge the data that we have here into a single dataframe and export it as a csv. 

In [25]:
print(prices.columns)
print(prices_indicies.columns)

Index(['BIL', 'BND', 'GLD', 'HYG', 'IEF', 'IWM', 'LQD', 'QQQ', 'SPY', 'TIP',
       'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV',
       'XLY'],
      dtype='str', name='Ticker')
Index(['CL=F', 'DX-Y.NYB', 'GC=F', 'HG=F', 'HYG', 'LQD', 'NG=F', 'SHY', 'TLT',
       '^FTSE', '^GSPC', '^HSI', '^MOVE', '^TNX', '^VIX', '^VXN'],
      dtype='str', name='Ticker')


In [26]:
df_merged = pd.merge(prices, prices_indicies, on="Date", how="outer")
df_merged.dropna(how='all',inplace=True)
df_merged.head()

Ticker,BIL,BND,GLD,HYG_x,IEF,IWM,LQD_x,QQQ,SPY,TIP,...,NG=F,SHY,TLT_y,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
Date,,,,,,,,,,,,,,,,,,,,,
1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


In [27]:
df_merged.to_csv('yfinance_data.csv', index_label='date')